# **Yoga se hi hoga**




In [ ]:
# Yoga Asana Recognition using LRCN (Video Classification)

!pip install opencv-python-headless moviepy

# Install libraries.
!pip install tensorflow opencv-contrib-python youtube-dl moviepy pydot
!pip install git+https://github.com/TahaAnwar/pafy.git#egg=pafy

In [ ]:

# Import libraries
import os
import cv2
import pafy
import math
import random
import numpy as np
import datetime as dt
import tensorflow as tf
from collections import deque
import matplotlib.pyplot as plt

from moviepy.editor import *
%matplotlib inline

from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import *
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.layers import TimeDistributed, Conv2D, MaxPooling2D, Dropout, Flatten, LSTM, Dense
from moviepy.editor import VideoFileClip


from tensorflow.keras.applications.mobilenet_v2 import preprocess_input 

And will set `Numpy`, `Python`, and `Tensorflow` seeds to get consistent results on every execution.

In [ ]:
seed_constant = 27
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !wget "https://www.crcv.ucf.edu/data/UCF50.rar" -O "/content/drive/MyDrive/Video Class/Data/UCF50.rar" --no-check-certificate

In [ ]:
# !pip install unrar

In [ ]:
# !unrar x "/content/drive/MyDrive/Video Class/Data/UCF50.rar" "/content/drive/MyDrive/destination_folder/"

In [ ]:
# config
IMAGE_HEIGHT, IMAGE_WIDTH = 128, 128
SEQUENCE_LENGTH = 20
DATASET_DIR = "/content/drive/MyDrive/NN/Data/Yoga_videos"

In [ ]:
# detect class labels 
CLASSES_LIST = sorted([f for f in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, f))])
print("Detected Yoga Asanas:", CLASSES_LIST)

In [ ]:
# frame extraction
def frames_extraction(video_path):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_window = max(int(frames_count / SEQUENCE_LENGTH), 1)

    for frame_counter in range(SEQUENCE_LENGTH):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_window)
        success, frame = video_reader.read()
        if not success:
            break
        resized = cv2.resize(frame, (IMAGE_WIDTH, IMAGE_HEIGHT))
        normalized = resized / 255.0
        frames_list.append(normalized)

    video_reader.release()
    return frames_list

In [ ]:
def augment_frame(frame):
    # Simple flip/rotate example
    if np.random.rand() < 0.5:
        frame = cv2.flip(frame, 1)  # horizontal flip
    angle = np.random.randint(-15, 15)
    M = cv2.getRotationMatrix2D((IMAGE_WIDTH//2, IMAGE_HEIGHT//2), angle, 1)
    frame = cv2.warpAffine(frame, M, (IMAGE_WIDTH, IMAGE_HEIGHT))
    return frame


In [ ]:
# data creation
from collections import Counter
def create_dataset():
    features, labels, video_paths = [], [], []

    for class_index, class_name in enumerate(CLASSES_LIST):
        print(f"Processing: {class_name}")
        class_dir = os.path.join(DATASET_DIR, class_name)
        video_files = [f for f in os.listdir(class_dir) if f.endswith(('.mp4', '.avi', '.mov'))]

        for video_file in video_files:
            video_path = os.path.join(class_dir, video_file)
            frames = frames_extraction(video_path)
            if len(frames) == SEQUENCE_LENGTH:
                features.append(frames)
                labels.append(class_index)
                video_paths.append(video_path)
    print("Label Distribution:", Counter(labels))
    return np.array(features), to_categorical(labels), video_paths

features, labels, video_paths = create_dataset()
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.25, random_state=42, shuffle=True)


In [ ]:
import matplotlib.pyplot as plt
unique, counts = np.unique(np.argmax(labels, axis=1), return_counts=True)
plt.bar([CLASSES_LIST[i] for i in unique], counts)
plt.title("Number of Videos per Asana")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# build LRCN model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Input, TimeDistributed, LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
def create_LRCN_model():
    # model = Sequential()

    # model.add(TimeDistributed(Conv2D(32, (3,3), activation='relu', padding='same'),
    #                           input_shape=(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, 3)))
    # model.add(TimeDistributed(MaxPooling2D((2,2))))
    # model.add(TimeDistributed(Dropout(0.25)))

    # model.add(TimeDistributed(Conv2D(64, (3,3), activation='relu', padding='same')))
    # model.add(TimeDistributed(MaxPooling2D((2,2))))
    # model.add(TimeDistributed(Dropout(0.25)))

    # model.add(TimeDistributed(Flatten()))
    # model.add(LSTM(64))
    # model.add(Dense(len(CLASSES_LIST), activation='softmax'))

    # model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    # model.summary()
    base_model = MobileNetV2(include_top=False, weights='imagenet', input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3))
    base_model.trainable = False

    model = Sequential()
    model.add(Input(shape=(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, 3)))
    model.add(TimeDistributed(base_model))
    model.add(TimeDistributed(GlobalAveragePooling2D()))
    model.add(LSTM(64))
    model.add(Dropout(0.5))
    model.add(Dense(len(CLASSES_LIST), activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    model.summary()
    return model

model = create_LRCN_model()

In [ ]:
# training
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(X_train, y_train,
                    epochs=50,
                    batch_size=4,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    shuffle=True)

train_preds = model.predict(X_train)
train_acc = np.mean(np.argmax(train_preds, axis=1) == np.argmax(y_train, axis=1))
print("Training Accuracy:", train_acc)

In [ ]:
# evaluation
val_loss, val_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {val_acc:.2f}, Loss: {val_loss:.2f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES_LIST, yticklabels=CLASSES_LIST, cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_true, y_pred, target_names=CLASSES_LIST))


In [ ]:
SEQUENCE_LENGTH = 20
IMAGE_WIDTH = 128
IMAGE_HEIGHT = 128


In [ ]:
# predict and overlay on video

def predict_on_video(video_path, output_path, model):
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Input video not found: {video_path}")

    video_reader = cv2.VideoCapture(video_path)
    width = int(video_reader.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video_reader.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = video_reader.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 25

    video_writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    frames_queue = deque(maxlen=SEQUENCE_LENGTH)
    display_text = ""

    while video_reader.isOpened():
        success, frame = video_reader.read()
        if not success:
            break

        resized = cv2.resize(frame, (IMAGE_WIDTH, IMAGE_HEIGHT))

        # using MobileNetV2 preprocessing:
        normalized = preprocess_input(resized.astype(np.float32))

        frames_queue.append(normalized)

        if len(frames_queue) == SEQUENCE_LENGTH:
            input_batch = np.expand_dims(frames_queue, axis=0)
            prediction = model.predict(input_batch, verbose=0)[0]
            class_index = np.argmax(prediction)
            predicted_class_name = CLASSES_LIST[class_index]
            confidence = prediction[class_index]

            # confidence label
            if confidence >= 0.85:
                quality = "Good"
            elif confidence >= 0.60:
                quality = "Average"
            else:
                quality = "Poor"

            display_text = f"{predicted_class_name} ({confidence*100:.1f}%) - {quality}"

        # label on the frame
        cv2.putText(frame, display_text, (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
        video_writer.write(frame)

    video_reader.release()
    video_writer.release()
    print(f"Prediction saved to: {output_path}")



In [ ]:
# example
input_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Padmasana /VID20250127150419.mp4"
output_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Predicted_output_pad_charul.mp4"
predict_on_video(input_video, output_video, model)

In [ ]:
# output video
VideoFileClip(output_video, audio=False).ipython_display()

In [ ]:
# unknown data
input_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/New Data/Cobra Pose.mp4"
output_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Predicted_output_bhuj_new_data.mp4"
predict_on_video(input_video, output_video, model)

In [ ]:
# view video
VideoFileClip(output_video, audio=False).ipython_display()

In [ ]:
# ex2
input_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/bhujangasana/VID20250127150317.mp4"
output_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Predicted_output_bhuj_charul.mp4"
predict_on_video(input_video, output_video, model)

In [ ]:
VideoFileClip(output_video, audio=False).ipython_display()

In [ ]:
# ex3
input_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/trikonasana/VID20250127150112.mp4"
output_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Predicted_output_tri_charul.mp4"
predict_on_video(input_video, output_video, model)

In [ ]:
VideoFileClip(output_video, audio=False).ipython_display()

In [ ]:
# ex4
input_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/adho mukha virasana/VID20250127151802.mp4"
output_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Predicted_output_adho_vir_suvidh.mp4"
predict_on_video(input_video, output_video, model)

In [ ]:
VideoFileClip(output_video, audio=False).ipython_display()

In [ ]:
# ex5
input_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Padmasana /VID20250127144731.mp4"
output_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Predicted_output_pad.mp4"
predict_on_video(input_video, output_video, model)

In [ ]:
VideoFileClip(output_video, audio=False).ipython_display()

In [ ]:
input_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/bhujangasana/VID20250127135640.mp4"
output_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Predicted_output_bhuj.mp4"
predict_on_video(input_video, output_video, model)

In [ ]:
VideoFileClip(output_video, audio=False).ipython_display()

In [ ]:
input_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/shavasana/VID20250127135436.mp4"
output_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Predicted_output_shav.mp4"
predict_on_video(input_video, output_video, model)

In [ ]:
VideoFileClip(output_video, audio=False).ipython_display()

In [ ]:
input_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/trikonasana/VID20250127135303.mp4"
output_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Predicted_output_tri.mp4"
predict_on_video(input_video, output_video, model)

In [ ]:
VideoFileClip(output_video, audio=False).ipython_display()

In [ ]:
input_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/adho mukha savasana/VID20250127135543.mp4"
output_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Predicted_output_adho_sav.mp4"
predict_on_video(input_video, output_video, model)

In [ ]:
VideoFileClip(output_video, audio=False).ipython_display()

In [ ]:
input_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/adho mukha virasana/VID20250127135405.mp4"
output_video = "/content/drive/MyDrive/NN/Data/Yoga_videos/Predicted_output_adho_vir.mp4"
predict_on_video(input_video, output_video, model)

In [ ]:
VideoFileClip(output_video, audio=False).ipython_display()